# 🤖 Smart Personal Assistant — Multi-Agent System

**Architecture:**
```
User Query
    │
    ▼
Router Agent  ──► identifies intents, selects specialized agents  (OpenAI GPT)
    │
    ├──► Weather Agent      (Open-Meteo API)
    ├──► Crypto Agent       (CoinGecko API)
    ├──► Currency Agent     (ExchangeRate API)
    ├──► Joke Agent         (JokeAPI)
    └──► Quote Agent        (Quotable API)
              │
              ▼
        Combiner Agent  ──► merges all results  (OpenAI GPT)
              │
              ▼
        Report Generator ──► saves timestamped .txt report
```

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install requests openai --quiet
print('✅ Dependencies installed.')

## 🔑 Cell 2 — API Key Setup

In [ ]:
import os
from getpass import getpass

# Enter your OpenAI API key (used by Router & Combiner agents)
OPENAI_API_KEY = getpass('🔑 Enter your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('✅ API key set.')

## 🏗️ Cell 3 — Shared State & Utilities

In [ ]:
import json
import time
import requests
from openai import OpenAI
from datetime import datetime
from typing import Optional

# ─── OpenAI client (singleton) ───────────────────────────────────────────────
def get_openai_client() -> OpenAI:
    return OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# ─── Shared State ────────────────────────────────────────────────────────────
class SharedState:
    """Central state store passed between all agents."""
    def __init__(self, user_query: str):
        self.user_query      = user_query
        self.selected_agents = []    # set by Router Agent
        self.agent_results   = {}    # raw API outputs keyed by agent name
        self.final_response  = ''    # set by Combiner Agent
        self.timestamp       = datetime.now()
        self.errors          = {}    # per-agent errors

# ─── Retry Helper ────────────────────────────────────────────────────────────
def fetch_with_retry(url: str, params: dict = None,
                     max_retries: int = 3, timeout: int = 10) -> Optional[dict]:
    """GET request with exponential-backoff retries."""
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.RequestException as e:
            print(f'  ⚠️  Attempt {attempt}/{max_retries} failed: {e}')
            if attempt < max_retries:
                time.sleep(2 ** attempt)   # 2 s, 4 s, 8 s
    return None

print('✅ SharedState & retry helper defined.')

## 🌤️ Cell 4 — Specialized Agents

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. WEATHER AGENT
# ─────────────────────────────────────────────────────────────────────────────
class WeatherAgent:
    NAME    = 'weather'
    API_URL = 'https://api.open-meteo.com/v1/forecast'
    WMO_CODES = {
        0:'Clear sky', 1:'Mainly clear', 2:'Partly cloudy', 3:'Overcast',
        45:'Foggy', 48:'Icy fog', 51:'Light drizzle', 53:'Moderate drizzle',
        55:'Dense drizzle', 61:'Slight rain', 63:'Moderate rain', 65:'Heavy rain',
        71:'Slight snow', 73:'Moderate snow', 75:'Heavy snow',
        80:'Slight showers', 81:'Moderate showers', 82:'Violent showers',
        95:'Thunderstorm', 96:'Thunderstorm w/ hail', 99:'Thunderstorm w/ heavy hail',
    }

    def run(self, state: SharedState) -> dict:
        print('  🌤️  WeatherAgent: fetching...')
        params = {
            'latitude': 13.08, 'longitude': 80.27,
            'current_weather': 'true',
            'hourly': 'precipitation_probability',
            'forecast_days': 1,
        }
        data = fetch_with_retry(self.API_URL, params=params)
        if not data:
            state.errors[self.NAME] = 'Weather API unreachable after retries.'
            return {'error': state.errors[self.NAME]}

        cw          = data.get('current_weather', {})
        wmo         = cw.get('weathercode', -1)
        description = self.WMO_CODES.get(wmo, f'Code {wmo}')
        rain_probs  = data.get('hourly', {}).get('precipitation_probability', [])
        max_rain    = max(rain_probs) if rain_probs else 'N/A'

        result = {
            'location'             : 'Chennai, India (13.08°N, 80.27°E)',
            'temperature_celsius'  : cw.get('temperature'),
            'windspeed_kmh'        : cw.get('windspeed'),
            'condition'            : description,
            'rain_probability_pct' : max_rain,
            'is_day'               : bool(cw.get('is_day', 1)),
            'raw'                  : cw,
        }
        state.agent_results[self.NAME] = result
        print(f'     ✅ {result["temperature_celsius"]}°C, {description}')
        return result


# ─────────────────────────────────────────────────────────────────────────────
# 2. CRYPTO AGENT
# ─────────────────────────────────────────────────────────────────────────────
class CryptoAgent:
    NAME    = 'crypto'
    API_URL = 'https://api.coingecko.com/api/v3/simple/price'

    def run(self, state: SharedState) -> dict:
        print('  💰  CryptoAgent: fetching...')
        params = {'ids': 'bitcoin,ethereum,dogecoin', 'vs_currencies': 'usd,inr'}
        data   = fetch_with_retry(self.API_URL, params=params)
        if not data:
            state.errors[self.NAME] = 'CoinGecko API unreachable after retries.'
            return {'error': state.errors[self.NAME]}

        def fmt(coin):
            d = data.get(coin, {})
            return {'usd': d.get('usd', 'N/A'), 'inr': d.get('inr', 'N/A')}

        result = {
            'bitcoin' : fmt('bitcoin'),
            'ethereum': fmt('ethereum'),
            'dogecoin': fmt('dogecoin'),
            'raw'     : data,
        }
        state.agent_results[self.NAME] = result
        btc_usd = result['bitcoin']['usd']
        print(f'     ✅ BTC ${btc_usd:,}' if isinstance(btc_usd, (int, float)) else f'     ✅ BTC {btc_usd}')
        return result


# ─────────────────────────────────────────────────────────────────────────────
# 3. CURRENCY AGENT
# ─────────────────────────────────────────────────────────────────────────────
class CurrencyAgent:
    NAME    = 'currency'
    API_URL = 'https://open.er-api.com/v6/latest/USD'
    POPULAR = ['INR','EUR','GBP','JPY','AUD','CAD','SGD','AED','CHF']

    def run(self, state: SharedState) -> dict:
        print('  💱  CurrencyAgent: fetching...')
        data = fetch_with_retry(self.API_URL)
        if not data:
            state.errors[self.NAME] = 'ExchangeRate API unreachable after retries.'
            return {'error': state.errors[self.NAME]}

        rates  = data.get('rates', {})
        result = {
            'base'          : 'USD',
            'popular_rates' : {k: rates[k] for k in self.POPULAR if k in rates},
            'last_update'   : data.get('time_last_update_utc', 'N/A'),
            'raw'           : rates,
        }
        state.agent_results[self.NAME] = result
        print(f'     ✅ 1 USD = {result["popular_rates"].get("INR", "?")} INR')
        return result


# ─────────────────────────────────────────────────────────────────────────────
# 4. JOKE AGENT
# ─────────────────────────────────────────────────────────────────────────────
class JokeAgent:
    NAME    = 'joke'
    API_URL = 'https://v2.jokeapi.dev/joke/Any'

    def run(self, state: SharedState) -> dict:
        print('  😂  JokeAgent: fetching...')
        data = fetch_with_retry(self.API_URL)
        if not data:
            state.errors[self.NAME] = 'JokeAPI unreachable after retries.'
            return {'error': state.errors[self.NAME]}

        if data.get('type') == 'single':
            joke_text = data.get('joke', '')
        else:
            joke_text = f"{data.get('setup', '')} ... {data.get('delivery', '')}"

        result = {
            'joke'    : joke_text,
            'category': data.get('category', 'General'),
            'flags'   : data.get('flags', {}),
            'raw'     : data,
        }
        state.agent_results[self.NAME] = result
        print(f'     ✅ Joke fetched ({result["category"]})')
        return result


# ─────────────────────────────────────────────────────────────────────────────
# 5. QUOTE AGENT
# ─────────────────────────────────────────────────────────────────────────────
class QuoteAgent:
    NAME    = 'quote'
    API_URL = 'https://api.quotable.io/random'

    def run(self, state: SharedState) -> dict:
        print('  💬  QuoteAgent: fetching...')
        data = fetch_with_retry(self.API_URL)
        if not data:
            state.errors[self.NAME] = 'Quotable API unreachable after retries.'
            return {'error': state.errors[self.NAME]}

        result = {
            'quote' : data.get('content', ''),
            'author': data.get('author', 'Unknown'),
            'tags'  : data.get('tags', []),
            'raw'   : data,
        }
        state.agent_results[self.NAME] = result
        print(f'     ✅ Quote by {result["author"]}')
        return result


# ─── Registry ────────────────────────────────────────────────────────────────
AGENT_REGISTRY = {
    'weather' : WeatherAgent(),
    'crypto'  : CryptoAgent(),
    'currency': CurrencyAgent(),
    'joke'    : JokeAgent(),
    'quote'   : QuoteAgent(),
}

print('✅ All 5 specialized agents defined.')

## 🔀 Cell 5 — Router Agent (GPT-powered)

In [ ]:
class RouterAgent:
    """
    Uses OpenAI GPT to detect user intent and select one or more agents.
    Falls back to keyword matching if the LLM call fails.
    """

    SYSTEM_PROMPT = """You are a routing agent for a multi-agent assistant.
Given a user query, return ONLY a JSON array of agent names to activate.
Available agents: ["weather", "crypto", "currency", "joke", "quote"]

Routing rules:
- weather  → weather, temperature, rain, forecast, climate, hot, cold, sunny
- crypto   → bitcoin, ethereum, dogecoin, crypto, BTC, ETH, coin, price
- currency → USD, INR, EUR, exchange rate, convert, forex, rupee, dollar, currency
- joke     → joke, funny, laugh, humour, humor, entertain, cheer up
- quote    → quote, motivate, inspire, wisdom, saying, encouragement

Return ONLY valid JSON. Examples:
  ["weather"]
  ["crypto", "currency"]
  ["joke", "quote"]
If nothing matches, return: ["joke"]"""

    def route(self, state: SharedState) -> list:
        print('🔀 RouterAgent: analysing query with GPT...')
        try:
            client = get_openai_client()
            resp = client.chat.completions.create(
                model       = 'gpt-4o-mini',
                max_tokens  = 64,
                temperature = 0,
                messages    = [
                    {'role': 'system', 'content': self.SYSTEM_PROMPT},
                    {'role': 'user',   'content': state.user_query},
                ],
            )
            raw    = resp.choices[0].message.content.strip()
            raw    = raw.replace('```json', '').replace('```', '').strip()
            agents = json.loads(raw)
            agents = [a for a in agents if a in AGENT_REGISTRY]
            if not agents:
                agents = ['joke']
        except Exception as e:
            print(f'  ⚠️  GPT routing failed ({e}), using keyword fallback.')
            agents = self._keyword_fallback(state.user_query)

        state.selected_agents = agents
        print(f'  ✅ Selected agents: {agents}')
        return agents

    def _keyword_fallback(self, query: str) -> list:
        q     = query.lower()
        found = []
        if any(w in q for w in ['weather','rain','temperature','forecast','hot','cold','climate','sunny']):
            found.append('weather')
        if any(w in q for w in ['bitcoin','ethereum','dogecoin','crypto','btc','eth','coin','price']):
            found.append('crypto')
        if any(w in q for w in ['usd','inr','eur','exchange','convert','forex','rupee','dollar','currency']):
            found.append('currency')
        if any(w in q for w in ['joke','funny','laugh','humour','humor','entertain']):
            found.append('joke')
        if any(w in q for w in ['quote','motivat','inspir','wisdom','saying']):
            found.append('quote')
        return found if found else ['joke']


router = RouterAgent()
print('✅ RouterAgent (GPT-powered) defined.')

## 🔗 Cell 6 — Combiner Agent (GPT-powered)

In [ ]:
class CombinerAgent:
    """
    Uses OpenAI GPT to merge all agent outputs into one coherent final response.
    Falls back to plain text summary if the LLM call fails.
    """

    SYSTEM_PROMPT = """You are a final-response combiner for a multi-agent AI assistant.
You receive structured data from one or more specialized agents and must:
1. Directly answer the user's original question.
2. Present all relevant information clearly and concisely.
3. Use plain language — no JSON, no code blocks.
4. Remove redundancy; keep the response friendly and well-formatted with emojis where appropriate."""

    def combine(self, state: SharedState) -> str:
        print('🔗 CombinerAgent: merging results with GPT...')

        payload = {
            'user_query'   : state.user_query,
            'agent_outputs': state.agent_results,
            'errors'       : state.errors,
        }

        try:
            client = get_openai_client()
            resp = client.chat.completions.create(
                model      = 'gpt-4o-mini',
                max_tokens = 1024,
                messages   = [
                    {'role': 'system', 'content': self.SYSTEM_PROMPT},
                    {
                        'role'   : 'user',
                        'content': (
                            f'Original query: {state.user_query}\n\n'
                            f'Agent data:\n{json.dumps(payload, indent=2, default=str)}'
                        ),
                    },
                ],
            )
            response = resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'  ⚠️  GPT combine failed ({e}), using plain fallback.')
            response = self._plain_fallback(state)

        state.final_response = response
        print('  ✅ Final response generated.')
        return response

    def _plain_fallback(self, state: SharedState) -> str:
        lines = [f'Response to: "{state.user_query}"', '']
        for name, result in state.agent_results.items():
            lines.append(f'── {name.upper()} ──')
            if isinstance(result, dict) and 'error' not in result:
                for k, v in result.items():
                    if k != 'raw':
                        lines.append(f'  {k}: {v}')
            else:
                lines.append(f'  {result}')
            lines.append('')
        return '\n'.join(lines)


combiner = CombinerAgent()
print('✅ CombinerAgent (GPT-powered) defined.')

## 📄 Cell 7 — Report Generator

In [ ]:
REPORT_DIR = 'reports'
os.makedirs(REPORT_DIR, exist_ok=True)

def generate_report(state: SharedState) -> str:
    """Creates a timestamped .txt report and returns its path."""
    ts   = state.timestamp.strftime('%Y_%m_%d_%H_%M')
    path = os.path.join(REPORT_DIR, f'report_{ts}.txt')
    sep  = '=' * 70

    lines = [
        sep,
        '          SMART PERSONAL ASSISTANT — EXECUTION REPORT',
        sep,
        f'Execution Date & Time : {state.timestamp.strftime("%Y-%m-%d %H:%M:%S")}',
        f'User Query            : {state.user_query}',
        f'Agents Activated      : {", ".join(state.selected_agents)}',
        '',
        sep,
        'RAW API OUTPUTS',
        sep,
    ]

    for agent_name, result in state.agent_results.items():
        lines.append(f'\n[ {agent_name.upper()} AGENT ]')
        lines.append(json.dumps(result, indent=2, default=str))

    if state.errors:
        lines += ['', sep, 'ERRORS', sep]
        for agent_name, err in state.errors.items():
            lines.append(f'  {agent_name}: {err}')

    lines += ['', sep, 'FINAL COMBINED RESPONSE', sep, '', state.final_response, '']
    lines += [sep, 'END OF REPORT', sep]

    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print(f'📄 Report saved → {path}')
    return path

print('✅ Report generator defined.')

## 🎛️ Cell 8 — Workflow Visualizer

In [ ]:
def visualize_workflow(state: SharedState):
    """Prints an ASCII workflow diagram based on the executed state."""
    icons = {'weather':'🌤️ ','crypto':'💰 ','currency':'💱 ','joke':'😂 ','quote':'💬 '}
    print('\n' + '─' * 60)
    print('  WORKFLOW EXECUTION DIAGRAM')
    print('─' * 60)
    print(f'  User Query: "{state.user_query}"')
    print('       │')
    print('       ▼')
    print('  ┌──────────────┐')
    print('  │ Router Agent │  (OpenAI GPT-4o-mini)')
    print('  └──────┬───────┘')
    print('         │')
    for a in AGENT_REGISTRY:
        if a in state.errors:
            status = '❌ ERROR   '
        elif a in state.selected_agents:
            status = '✅ EXECUTED'
        else:
            status = '⏭️  SKIPPED '
        print(f'         ├── {status}  {icons.get(a, "🤖 ")}{a.capitalize()} Agent')
    print('         │')
    print('         ▼')
    print('  ┌────────────────┐')
    print('  │ Combiner Agent │  (OpenAI GPT-4o-mini)')
    print('  └──────┬─────────┘')
    print('         │')
    print('         ▼')
    print('  ┌──────────────────┐')
    print('  │ Report Generator │  (.txt + .json)')
    print('  └──────────────────┘')
    print('─' * 60 + '\n')

print('✅ Workflow visualizer defined.')

## 🚀 Cell 9 — Main Orchestrator

In [ ]:
def run_assistant(user_query: str) -> SharedState:
    """
    Full pipeline:
      Router → Specialized Agents → Combiner → Report
    Returns the final SharedState for inspection.
    """
    print('\n' + '═' * 60)
    print('  🤖 SMART PERSONAL ASSISTANT')
    print(f'  Query: "{user_query}"')
    print('═' * 60)

    # 1. Initialise shared state
    state = SharedState(user_query)

    # 2. Route
    router.route(state)

    # 3. Execute selected agents
    print('\n📡 Executing agents...')
    for agent_name in state.selected_agents:
        AGENT_REGISTRY[agent_name].run(state)

    # 4. Combine
    print()
    combiner.combine(state)

    # 5. Generate .txt report
    generate_report(state)

    # 6. Visualise workflow
    visualize_workflow(state)

    # 7. Print final answer
    print('\n' + '═' * 60)
    print('  💬 FINAL RESPONSE')
    print('═' * 60)
    print(state.final_response)
    print('═' * 60 + '\n')

    # 8. Persist JSON snapshot
    snapshot = {
        'query'          : state.user_query,
        'selected_agents': state.selected_agents,
        'agent_results'  : state.agent_results,
        'errors'         : state.errors,
        'final_response' : state.final_response,
        'timestamp'      : state.timestamp.isoformat(),
    }
    json_path = os.path.join(
        REPORT_DIR,
        f'snapshot_{state.timestamp.strftime("%Y_%m_%d_%H_%M")}.json'
    )
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(snapshot, f, indent=2, default=str)
    print(f'💾 JSON snapshot saved → {json_path}')

    return state

print('✅ Orchestrator defined. Ready to run!')

## ▶️ Cell 10 — Run Example Queries

In [ ]:
# ── Single query ──────────────────────────────────────────────────────────────
query = "What's the weather today and what is the Bitcoin price?"
state = run_assistant(query)

In [ ]:
# ── More example queries — uncomment one and run ──────────────────────────────

# state = run_assistant("Tell me a joke")
# state = run_assistant("Convert USD to INR")
# state = run_assistant("Give me a motivational quote and show Ethereum price")
# state = run_assistant("Will it rain today? Also show all crypto prices.")
# state = run_assistant("Show me the latest exchange rates and inspire me")

## 🗂️ Cell 11 — List All Saved Reports

In [ ]:
import glob

print('📁 TXT Reports:')
for f in sorted(glob.glob(os.path.join(REPORT_DIR, 'report_*.txt'))):
    print(f'  📄 {f}  ({os.path.getsize(f)} bytes)')

print('\n💾 JSON Snapshots:')
for f in sorted(glob.glob(os.path.join(REPORT_DIR, 'snapshot_*.json'))):
    print(f'  🗂️  {f}')

## 📖 Cell 12 — Read Latest Report

In [ ]:
reports = sorted(glob.glob(os.path.join(REPORT_DIR, 'report_*.txt')))
if reports:
    latest = reports[-1]
    print(f'Reading: {latest}\n')
    with open(latest, encoding='utf-8') as f:
        print(f.read())
else:
    print('No reports yet — run Cell 10 first.')

## 🎯 Cell 13 — Interactive Chat Loop

In [ ]:
print('🤖 Smart Personal Assistant — Interactive Mode')
print('Type your query and press Enter. Type "exit" to quit.\n')

while True:
    try:
        q = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Goodbye!')
        break
    if not q:
        continue
    if q.lower() in ('exit', 'quit', 'bye'):
        print('👋 Goodbye!')
        break
    run_assistant(q)